# RQ1 & RQ2 Data Analysis for READ-MAS

This notebook reproduces the statistical analysis (ANOVA / Kruskal-Wallis + post-hoc tests) for the two research questions covered by this evidence package:

- **RQ1**: Does using multiple agents (READ-MAS) compared to a single agent improve code generation (HumanEval, MBPP) and design quality (custom LLM-as-a-Judge, RAGAS faithfulness)?
- **RQ2**: Does Retrieval-Augmented Generation (RAG) improve READ-MAS design accuracy compared to the same agent without RAG (requirements corpus and DevBench corpus)?

It is a trimmed copy of `notebooks/data_analysis/data_analysis.ipynb` from the full READ-MAS repository, containing only the cells needed for these two questions. (RQ2 here is RQ5 in the parent dissertation; the dissertation's own RQ2/RQ3/RQ4 sections were removed as out of scope.)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import pingouin
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# The parent path to the benchmark experiment runs
EXPERIMENT_PATH = '../../benchmark_runs/'

# Column renames for LLM as a Judge benchmark runs
RENAME_BENCHMARK = {
  'runs/benchmark_runs/metrics.json:agent':                    'agent',
  'runs/benchmark_runs/metrics.json:model':                    'model',
  'runs/benchmark_runs/metrics.json:rag':                      'rag',
  'runs/benchmark_runs/metrics.json:DesignAccuracy [GEval]':   'design_accuracy',
  'DesignAccuracy [GEval]':                                    'design_accuracy',
  'runs/benchmark_runs/metrics.json:faithfulness (ragas)':     'ragas_faithfulness',
  'runs/benchmark_runs/metrics.json:RAGAS':                    'ragas',
}
# Columns to keep in the benchmark runs data frame
KEEP_BENCHMARK = ['agent', 'model', 'rag', 'design_accuracy', 'ragas_faithfulness', 'ragas']

# Column renames for code benchmark runs
RENAME_CODE_BENCHMARK = {
  'code.agent_name':                                          'agent',
  'code.model':                                               'model',
  'code.rag':                                                 'rag',
  'pass@1':                                                   'pass_at_1',
  'pass@1plus':                                               'pass_at_1plus',
  'code.dataset':                                             'benchmark',
  'runs/code_benchmark_runs/metrics.json:pass@1':             'pass_at_1',
  'runs/code_benchmark_runs/metrics.json:pass@1plus':         'pass_at_1plus',
}

# Columns to keep for code benchmark runs
KEEP_CODE_BENCHMARK = ['agent', 'model', 'rag', 'pass_at_1', 'pass_at_1plus', 'benchmark']

# LLM as a Judge and RAGAS metrics
BENCHMARK_METRICS = ['design_accuracy', 'ragas_faithfulness', 'ragas']

# HumanEval and MBPP metrics for code benchmarking
CODE_METRICS = ['pass_at_1', 'pass_at_1plus']

# Metrics without RAGAS for RQ2
NO_RAGAS_METRICS = ['design_accuracy']

# Mapping of custom benchmarks and metrics
BENCHMARK_COL_METRICS_MAP = {
  'benchmark': {
    'rename_cols': RENAME_BENCHMARK,
    'keep_cols': KEEP_BENCHMARK,
    'metrics': BENCHMARK_METRICS
  }
}

# Mapping of custom benchmarks and metrics for NO RAGAS use case
NO_RAGAS_BENCHMARK_COL_METRICS_MAP = {
  'benchmark': {
    'rename_cols': RENAME_BENCHMARK,
    'keep_cols': KEEP_BENCHMARK,
    'metrics': NO_RAGAS_METRICS
  }
}

# Mapping of coding benchmarks and metrics
CODE_BENCHMARK_COL_METRICS_MAP = {
  'humaneval': {
    'rename_cols': RENAME_CODE_BENCHMARK,
    'keep_cols': KEEP_CODE_BENCHMARK,
    'metrics': CODE_METRICS
  },
  'mbpp': {
    'rename_cols': RENAME_CODE_BENCHMARK,
    'keep_cols': KEEP_CODE_BENCHMARK,
    'metrics': CODE_METRICS
  }
}

# RQ1: Gemini - Custom benchmark, HumanEval and MBPP code benchmark experiment datasets
RQ1_BENCHMARK_FILES_GEMINI = {
    'benchmark': ('single_agent_benchmark_gemini.csv',
                  'read_agent_benchmark_gemini.csv'),
    'humaneval': ('single_agent_code_benchmark_humaneval_gemini.csv',
                'read_agent_code_benchmark_humaneval_gemini.csv'),
    'mbpp':      ('single_agent_code_benchmark_mbpp_gemini.csv',
                'read_agent_code_benchmark_mbpp_gemini.csv'),
}

# RQ1: GPT-5-mini - Custom benchmark, HumanEval and MBPP code benchmark experiment datasets
RQ1_BENCHMARK_FILES_OPENAI = {
    'benchmark': ('single_agent_benchmark_openai.csv', 'read_agent_benchmark_openai.csv'),
    'humaneval': ('single_agent_code_benchmark_humaneval_openai.csv',
                'read_agent_code_benchmark_humaneval_openai.csv'),
    'mbpp': ('single_agent_code_benchmark_mbpp_openai.csv',
              'read_agent_code_benchmark_mbpp_openai.csv')
}

# Columns and metrics map combining custom and code benchmarks.
FULL_COL_METRICS_MAP = BENCHMARK_COL_METRICS_MAP | CODE_BENCHMARK_COL_METRICS_MAP

# No RAGAS columns and metrics map combining custom and code benchmarks.
NO_RAGAS_COL_METRICS_MAP = NO_RAGAS_BENCHMARK_COL_METRICS_MAP | CODE_BENCHMARK_COL_METRICS_MAP

# RQ2: READ-MAS with vs without RAG, using the requirements corpus
RQ2_BENCHMARK_FILES = {
    'benchmark': ('read_agent_benchmark_gemini_rag.csv',
                  'read_agent_benchmark_gemini_no_rag.csv')
}

# RQ2: READ-MAS with vs without RAG, using the DevBench corpus for the RAG index
RQ2_INDEX_BENCHMARK_FILES = {
    'benchmark': ('read_agent_benchmark_gemini_index.csv',
                  'read_agent_benchmark_gemini_no_rag.csv'),
}


## Helper Functions for Experiment Data Loading and Performing ANOVA

In [ ]:
def load_experiment_dataset(filename: str, rename_cols: dict[str, str], keep_cols: list[str]) -> pd.DataFrame:
    """Loads an experiment CSV, filters out the baseline rows, renames and keeps only the needed columns."""
    df = pd.read_csv(EXPERIMENT_PATH + filename)
    df = df[df['typ'].isin(['branch_base', 'branch_commit'])]
    df = df.rename(columns=rename_cols)

    # Set agent to llm for LLM benchmark datasets
    if 'agent' not in df.columns:
      df['agent'] = 'llm'

    return df[keep_cols]


In [ ]:
def get_group_dfs(group_files: dict[str, tuple[str, ...]], group_col_metrics_map: dict[str, dict[str, str]]) -> list[dict]:
  """Builds a list of {benchmark, group_df, metrics} dicts for a given ANOVA/EDA grouping."""
  group_dfs = []
  for benchmark, files in group_files.items():
    config = group_col_metrics_map[benchmark]
    rename_cols = config['rename_cols']
    keep_cols = config['keep_cols']
    metrics = config['metrics']

    df_list = [
        load_experiment_dataset(f, rename_cols, keep_cols)
        for f in files
    ]

    group_df = pd.concat(df_list, ignore_index=True)

    group_dfs.append({'benchmark': benchmark, 'group_df': group_df, 'metrics': metrics})

  return group_dfs


In [ ]:
def perform_anova(benchmark: str, group_df: pd.DataFrame, metrics: list[str], group: str):
    """Runs ANOVA assumption checks, ANOVA/Kruskal-Wallis, and post hoc tests; prints Markdown tables."""
    group_df = group_df.copy()
    for metric in metrics:
        group_df[metric] = pd.to_numeric(group_df[metric], errors='coerce')

    print(f"\n# Benchmark: {benchmark}")

    for metric in metrics:
        print(f"\n## Metric: {metric}")

        norm = pingouin.normality(data=group_df, dv=metric, group=group, method='shapiro')
        print("\n#### Shapiro-Wilk Normality Test")
        print(f"*p > 0.05 indicates normality*")
        print(norm[['W', 'pval', 'normal']].to_markdown(index=False))

        eq_var = pingouin.homoscedasticity(data=group_df, dv=metric, group=group, method='levene')
        print("\n#### Levene Homoscedasticity Test")
        print(f"*p > 0.05 indicates equal variances*")
        print(eq_var[['W', 'pval', 'equal_var']].to_markdown(index=False))

        normal = norm['normal'].all()
        equal_var = eq_var['equal_var'].iloc[0]

        if not normal:
            kw = pingouin.kruskal(data=group_df, dv=metric, between=group)
            print(f"\n#### Kruskal-Wallis Test (Non-Parametric)")
            print(kw.to_markdown(index=False))

            medians = group_df.groupby(group)[metric].median().reset_index()
            medians.columns = [group, 'Median']
            print(f"\n#### Group Medians")
            print(medians.to_markdown(index=False))

        elif not equal_var:
            aov = pingouin.welch_anova(data=group_df, dv=metric, between=group)
            print(f"\n#### Welch's ANOVA")
            print(aov.to_markdown(index=False))
            print(f"\n**Summary:** F({aov.iloc[0]['ddof1']:.0f}, {aov.iloc[0]['ddof2']:.2f}) = {aov.iloc[0]['F']:.4f}, p = {aov.iloc[0]['p_unc']:.4f}")

            gh = pingouin.pairwise_gameshowell(data=group_df, dv=metric, between=group)
            print(f"\n#### Games-Howell Post Hoc Test")
            print(gh.to_markdown(index=False))

        else:
            aov = pingouin.anova(data=group_df, dv=metric, between=group, detailed=True)
            print(f"\n#### One-way ANOVA")
            print(aov.to_markdown(index=False))
            print(f"\n**Summary:** F({aov.iloc[0]['DF']}, {aov.iloc[1]['DF']}) = {aov.iloc[0]['F']:.4f}, p = {aov.iloc[0]['p_unc']:.4f}")

            tukey = pingouin.pairwise_tukey(data=group_df, dv=metric, between=group)
            print(f"\n#### Tukey HSD Post Hoc Test")
            print(tukey.to_markdown(index=False))


## RQ1: Compare Single and Multi Agents

Does using multiple agents compared to a single agent improve the performance of the generated code, as measured by the HumanEval and MBPP benchmarks, and the design, as measured by custom LLM-as-a-Judge and RAGAS faithfulness metrics?

In [ ]:
# RQ1 grouped datasets - Gemini
rq1_group_gemini_dfs = get_group_dfs(RQ1_BENCHMARK_FILES_GEMINI, FULL_COL_METRICS_MAP)

for group in rq1_group_gemini_dfs:
  print(f"Benchmark {group['benchmark']}  Metrics {group['metrics']}\n")
  print(f"RQ1 combined group dataset (Gemini) for {group['benchmark']}: \n")
  print(group['group_df'].to_markdown(index=False))
  print("\n\n")

# RQ1 grouped datasets - OpenAI
rq1_group_openai_dfs = get_group_dfs(RQ1_BENCHMARK_FILES_OPENAI, FULL_COL_METRICS_MAP)

for group in rq1_group_openai_dfs:
  print(f"RQ1 combined group dataset (OpenAI) for {group['benchmark']}: \n")
  print(group['group_df'].to_markdown(index=False))
  print("\n\n")


### Single vs Multi-agent using Gemini-2.5-Flash

In [ ]:
# RQ1 ANOVA analysis for experiments that used Gemini-2.5-flash model
for group in rq1_group_gemini_dfs:
  perform_anova(group['benchmark'], group['group_df'], group['metrics'], 'agent')


### Single vs Multi-agent using GPT-5-mini

In [ ]:
# RQ1 ANOVA analysis for experiments that used OpenAI GPT-5-mini model
for group in rq1_group_openai_dfs:
  perform_anova(group['benchmark'], group['group_df'], group['metrics'], 'agent')


## RQ2: Compare READ-MAS with and without RAG

How does the new multi-agent framework with RAG compare to the framework without RAG in producing more accurate designs as measured by the custom LLM-as-a-Judge metric?

In [ ]:
# RQ2 grouped datasets (requirements corpus RAG index)
rq2_group_dfs = get_group_dfs(RQ2_BENCHMARK_FILES, NO_RAGAS_COL_METRICS_MAP)

for group in rq2_group_dfs:
  print(f"RQ2 combined group dataset for {group['benchmark']}: \n")
  print(group['group_df'].to_markdown(index=False))
  print("\n\n")

# RQ2 grouped datasets (DevBench corpus RAG index)
rq2_devbench_group_dfs = get_group_dfs(RQ2_INDEX_BENCHMARK_FILES, NO_RAGAS_COL_METRICS_MAP)

for group in rq2_devbench_group_dfs:
  print(f"RQ2 combined group dataset for {group['benchmark']}: \n")
  print(group['group_df'].to_markdown(index=False))
  print("\n\n")


### RAG vs no-RAG using the Requirements Dataset

In [ ]:
# Run ANOVA on cleaned data
for group in rq2_group_dfs:
    perform_anova(group['benchmark'], group['group_df'], group['metrics'], 'rag')


### RAG vs no-RAG using the DevBench Dataset

In [ ]:
# Run ANOVA on cleaned data that uses the DevBench RAG index
for group in rq2_devbench_group_dfs:
    perform_anova(group['benchmark'], group['group_df'], group['metrics'], 'rag')


### Effect of RAG on Design Accuracy (EDA)

In [ ]:
# Box and strip plots for design_accuracy with vs without RAG
df_bm_rag = rq2_group_dfs[0]['group_df'].copy()
df_bm_rag['design_accuracy'] = pd.to_numeric(df_bm_rag['design_accuracy'], errors='coerce')
df_bm_rag['RAG'] = df_bm_rag['rag'].map({True: 'RAG enabled', False: 'RAG disabled'})

df_bm_rag_specific = rq2_devbench_group_dfs[0]['group_df'].copy()
df_bm_rag_specific['design_accuracy'] = pd.to_numeric(df_bm_rag_specific['design_accuracy'], errors='coerce')
df_bm_rag_specific['RAG'] = df_bm_rag_specific['rag'].map({True: 'RAG enabled', False: 'RAG disabled'})

order_rag = ['RAG disabled', 'RAG enabled']
palette_rag = {'RAG enabled': '#4C72B0', 'RAG disabled': '#DD8452'}
sub_titles = ['RAG from Requirements', 'RAG from DevBench']

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, data, sub_title in zip(axes, [df_bm_rag, df_bm_rag_specific], sub_titles):
    sns.boxplot(data=data, x='RAG', y='design_accuracy', order=order_rag,
                palette=palette_rag, width=0.35, fliersize=0, ax=ax)
    sns.stripplot(data=data, x='RAG', y='design_accuracy', order=order_rag,
                palette=palette_rag, size=6, jitter=True, alpha=0.6, ax=ax)

    for i, label in enumerate(order_rag):
        vals = data[data['RAG'] == label]['design_accuracy']
        ax.text(i, vals.mean() + 0.002, f'M={vals.mean():.3f}', ha='center',
                fontsize=9, color='black', fontweight='bold')

    ax.set_title(sub_title, fontweight='bold')
    ax.set_ylabel('Design Accuracy [GEval]')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
    ax.set_xlabel('')

plt.suptitle('Design Accuracy: RAG enabled vs disabled\n(read_agent / gemini-2.5-flash)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
